In [1]:
import os

In [2]:
%pwd

'd:\\Project\\Cheast Disease Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Project\\Cheast Disease Classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        return data_ingestion_config

In [8]:
import os
import zipfile
import gdown
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size


In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self) -> None:
        """
        Download file from Google Drive
        """
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = Path(self.config.local_data_file)
            os.makedirs(self.config.root_dir, exist_ok=True)
            logger.info(f"Downloading file from :[{dataset_url}] into :[{zip_download_dir}]")
            
            file_id = dataset_url.split('/')[-2]
            prefix = "https://drive.google.com/uc?export=download&id="
            gdown.download(prefix + file_id, str(zip_download_dir))
            
            logger.info(f"Downloaded file from :[{dataset_url}] into :[{zip_download_dir}] of size :[{get_size(zip_download_dir)}]")
            
        except Exception as e:
            raise e
        
    def extract_zip_file(self) -> None:
        """
        Extract zip file
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-20 13:07:43,633: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-07-20 13:07:43,635: INFO: common]: yaml file: params.yaml loaded successfully
[2026-07-20 13:07:43,637: INFO: common]: created directory at: artifacts
[2026-07-20 13:07:43,638: INFO: common]: created directory at: artifacts/data_ingestion
[2026-07-20 13:07:43,639: INFO: 1156647531]: Downloading file from :[https://drive.google.com/file/d/1nY4Xm4GekPf0ZWRuFCfS43jXOfiR8LAc/view?usp=sharing] into :[artifacts\data_ingestion\data.zip]


Downloading...
From (original): https://drive.google.com/uc?export=download&id=1nY4Xm4GekPf0ZWRuFCfS43jXOfiR8LAc
From (redirected): https://drive.google.com/uc?export=download&id=1nY4Xm4GekPf0ZWRuFCfS43jXOfiR8LAc&confirm=t&uuid=f168f38a-fc52-493a-90e6-728fd79178c1
To: d:\Project\Cheast Disease Classification\artifacts\data_ingestion\data.zip
100%|██████████| 49.0M/49.0M [00:01<00:00, 25.8MB/s]

[2026-07-20 13:07:48,002: INFO: 1156647531]: Downloaded file from :[https://drive.google.com/file/d/1nY4Xm4GekPf0ZWRuFCfS43jXOfiR8LAc/view?usp=sharing] into :[artifacts\data_ingestion\data.zip] of size :[~ 47894 KB]
